# bb_pipeline PyTorch migration verification

Runs detection on one HD video and one feeder video in **four** environments:

| # | Video | Env | Localizer | Output |
|---|-------|-----|-----------|--------|
| 1 | HD | conda | heatmap | bb_binary repo |
| 2 | Feeder (RPi) | conda | POLO torchscript | parquet |
| 3 | HD | Docker | heatmap | bb_binary repo |
| 4 | Feeder (RPi) | Docker | POLO torchscript | parquet |

Both modes call the same job functions used in production (`bb_hpc.src.jobfunctions`), so success here means the production code path works end-to-end.

**Before running:** edit `HD_VIDEO_PATH` and `FEEDER_VIDEO_PATH` in the setup cell.

## Setup & environment check

In [1]:
# Edit these paths for your machine.
HD_VIDEO_PATH     = "/mnt/share/beesbook2025/20250801/cam-0/cam-0_20250801T090003.300824.299Z--20250801T090103.222699.929Z.mp4"
FEEDER_VIDEO_PATH = "/mnt/share/beesbook2025/pi/20250901/outdoorcam/outdoorcam_2025-09-01-16-54-06.h264"
OUT_ROOT          = "/home/jdavidson/pytorch_migration_test_runs"
DOCKER_IMAGE      = "jacobdavidson/beesbook:latest"
BB_HPC_DIR        = "/mnt/share/jacob/bb_hpc"   # same path on host and in Docker (bind is identity)

import os, sys, uuid, datetime, importlib.util, glob, json

# --- environment sanity check (conda kernel) -------------------------------
import torch
print(f"torch            : {torch.__version__}")
print(f"cuda available   : {torch.cuda.is_available()}")
print(f"cuda device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"cuda device[0]   : {torch.cuda.get_device_name(0)}")

assert importlib.util.find_spec("tensorflow") is None, \
    "tensorflow is still importable in this conda env!"
print("tensorflow       : not installed ✓")

model_dir = os.path.join(os.environ["CONDA_PREFIX"], "pipeline_models")
print(f"\nmodel files in {model_dir}:")
for f in sorted(glob.glob(os.path.join(model_dir, "*"))):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f):50s}  {size_mb:7.2f} MB")

# --- video paths exist? ----------------------------------------------------
for label, p in [("HD", HD_VIDEO_PATH), ("feeder", FEEDER_VIDEO_PATH)]:
    exists = os.path.isfile(p)
    print(f"{label:8s} video exists: {exists}  ({p})")

# --- build per-run output paths -------------------------------------------
run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
OUT = {
    "hd_conda":    os.path.join(OUT_ROOT, run_id, "hd_conda"),
    "polo_conda":  os.path.join(OUT_ROOT, run_id, "polo_conda"),
    "hd_docker":   os.path.join(OUT_ROOT, run_id, "hd_docker"),
    "polo_docker": os.path.join(OUT_ROOT, run_id, "polo_docker"),
}
print(f"\nrun_id: {run_id}")
for k, v in OUT.items():
    print(f"  {k:12s} -> {v}")

SCRIPT = f"{BB_HPC_DIR}/test/test_pipeline_single_video.py"

torch            : 2.11.0+cu128
cuda available   : True
cuda device count: 2
cuda device[0]   : NVIDIA GeForce GTX 1080 Ti
tensorflow       : not installed ✓

model files in /home/jdavidson/miniforge3/envs/beesbook/pipeline_models:
  decoder_2019_weights.pt                               14.28 MB
  detection_model_4.json                                 0.44 MB
  localizer_2019_attributes.json                         0.00 MB
  localizer_2019_weights.pt                              1.01 MB
  tracklet_model_8.json                                  0.89 MB
HD       video exists: False  (/mnt/share/beesbook2025/20250801/cam-0/cam-0_20250801T090003.300824.299Z--20250801T090103.222699.929Z.mp4)
feeder   video exists: False  (/mnt/share/beesbook2025/pi/20250901/outdoorcam/outdoorcam_2025-09-01-16-54-06.h264)

run_id: 20260415-121449-4f59b8
  hd_conda     -> /home/jdavidson/pytorch_migration_test_runs/20260415-121449-4f59b8/hd_conda
  polo_conda   -> /home/jdavidson/pytorch_migration_test_runs/20

/home/jdavidson/miniforge3/envs/beesbook/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce GTX 1080 Ti which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/jdavidson/miniforge3/envs/beesbook/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU1 NVIDIA GeForce GTX 1080 Ti which is of compute capability (CC) 6.1.
The 

## Test 1 — HD video, conda env

In [ ]:
!python -u "{SCRIPT}" --video "{HD_VIDEO_PATH}" --mode hd --out "{OUT['hd_conda']}" \
    --timestamp-format basler --video-file-type basler --num-threads 1

## Test 2 — Feeder video + POLO, conda env

In [ ]:
!python -u "{SCRIPT}" --video "{FEEDER_VIDEO_PATH}" --mode polo --out "{OUT['polo_conda']}" --clahe

## Test 3 — HD video, Docker image

Uses the same mount pattern as `bb_hpc/running_docker/detect_submit.py`: `/mnt/share → /mnt/share` (identity), so `HD_VIDEO_PATH` and `SCRIPT` paths work unchanged inside the container.

In [ ]:
docker_payload_hd = (
    "set -euo pipefail; "
    "source /opt/conda/etc/profile.d/conda.sh && conda activate beesbook; "
    "export LD_LIBRARY_PATH=/opt/conda/envs/beesbook/lib:${LD_LIBRARY_PATH:-}; "
    f"python -u {SCRIPT} --video '{HD_VIDEO_PATH}' --mode hd --out '{OUT['hd_docker']}' "
    "--timestamp-format basler --video-file-type basler --num-threads 1"
)
docker_cmd_hd = (
    f"docker run --rm --gpus all --runtime=nvidia "
    f"-v /mnt/share:/mnt/share "
    f"{DOCKER_IMAGE} bash -lc \"{docker_payload_hd}\""
)
print(docker_cmd_hd)
!{docker_cmd_hd}

## Test 4 — Feeder video + POLO, Docker image

In [ ]:
docker_payload_polo = (
    "set -euo pipefail; "
    "source /opt/conda/etc/profile.d/conda.sh && conda activate beesbook; "
    "export LD_LIBRARY_PATH=/opt/conda/envs/beesbook/lib:${LD_LIBRARY_PATH:-}; "
    f"python -u {SCRIPT} --video '{FEEDER_VIDEO_PATH}' --mode polo --out '{OUT['polo_docker']}' --clahe"
)
docker_cmd_polo = (
    f"docker run --rm --gpus all --runtime=nvidia "
    f"-v /mnt/share:/mnt/share "
    f"{DOCKER_IMAGE} bash -lc \"{docker_payload_polo}\""
)
print(docker_cmd_polo)
!{docker_cmd_polo}

## Verification — open outputs and count

- HD: bb_binary repo → iterate `.bbb` files, count frames and detections (`detectionsDP` = tagged, `detectionsBees` = untagged).
- POLO: parquet file → row count.

Pass bar: structural validity, not numerical equivalence (CUDA/cuDNN nondeterminism makes bit-exact comparison flaky).

In [ ]:
import pandas as pd
import bb_binary as bbb

def verify_hd(out_dir, label):
    if not os.path.isdir(out_dir):
        return dict(test=label, ok=False, reason="no output dir")
    bbb_files = sorted(glob.glob(os.path.join(out_dir, "**", "*.bbb"), recursive=True))
    if not bbb_files:
        return dict(test=label, ok=False, reason="no .bbb files")
    n_frames = n_dp = n_bees = 0
    for fc_path in bbb_files:
        with open(fc_path, "rb") as f:
            fc = bbb.FrameContainer.read(f)
        for frame in fc.frames:
            n_frames += 1
            which = frame.detectionsUnion.which()
            if which == "detectionsDP":
                n_dp += len(frame.detectionsUnion.detectionsDP)
            elif which == "detectionsBees":
                n_bees += len(frame.detectionsUnion.detectionsBees)
    ok = n_frames > 0 and (n_dp + n_bees) > 0
    return dict(test=label, ok=ok, bbb_files=len(bbb_files),
                frames=n_frames, detectionsDP=n_dp, detectionsBees=n_bees)

def verify_polo(out_dir, label):
    if not os.path.isdir(out_dir):
        return dict(test=label, ok=False, reason="no output dir")
    parquet_files = sorted(glob.glob(os.path.join(out_dir, "*.parquet")))
    if not parquet_files:
        return dict(test=label, ok=False, reason="no .parquet files")
    total_rows = 0
    for pq in parquet_files:
        total_rows += len(pd.read_parquet(pq))
    return dict(test=label, ok=total_rows > 0,
                parquet_files=len(parquet_files), rows=total_rows)

rows = [
    verify_hd  (OUT["hd_conda"],    "hd_conda"),
    verify_polo(OUT["polo_conda"],  "polo_conda"),
    verify_hd  (OUT["hd_docker"],   "hd_docker"),
    verify_polo(OUT["polo_docker"], "polo_docker"),
]
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print(f"tests passed: {int(df['ok'].sum())}/{len(df)}")
assert df["ok"].all(), "Some tests failed — see table above."

## Sanity similarity — conda vs Docker

Non-failing warning: the two environments should produce comparable detection counts (within ~10%). Large divergence is a red flag (different model weights, different CUDA behavior, etc.).

In [ ]:
def rel_diff(a, b):
    if max(a, b) == 0:
        return 0.0
    return abs(a - b) / max(a, b)

by_test = {r["test"]: r for r in rows}

# HD: compare total detections (dp + bees)
hd_c = by_test["hd_conda"].get("detectionsDP", 0) + by_test["hd_conda"].get("detectionsBees", 0)
hd_d = by_test["hd_docker"].get("detectionsDP", 0) + by_test["hd_docker"].get("detectionsBees", 0)
hd_rel = rel_diff(hd_c, hd_d)
flag_hd = " ⚠️" if hd_rel > 0.10 else ""
print(f"HD   detections  conda={hd_c:6d}  docker={hd_d:6d}  rel_diff={hd_rel:6.1%}{flag_hd}")

# POLO: compare parquet row counts
po_c = by_test["polo_conda"].get("rows", 0)
po_d = by_test["polo_docker"].get("rows", 0)
po_rel = rel_diff(po_c, po_d)
flag_po = " ⚠️" if po_rel > 0.10 else ""
print(f"POLO rows        conda={po_c:6d}  docker={po_d:6d}  rel_diff={po_rel:6.1%}{flag_po}")

if max(hd_rel, po_rel) > 0.10:
    print("\nWARNING: >10% relative difference between conda and Docker. Inspect outputs.")

## Sample detections — eyeball check

In [ ]:
def sample_hd(out_dir, label, n=5):
    bbb_files = sorted(glob.glob(os.path.join(out_dir, "**", "*.bbb"), recursive=True))
    if not bbb_files:
        print(f"[{label}] no .bbb files"); return
    print(f"\n[{label}] first {n} tagged detections from {os.path.basename(bbb_files[0])}")
    with open(bbb_files[0], "rb") as f:
        fc = bbb.FrameContainer.read(f)
    count = 0
    for frame in fc.frames:
        if count >= n: break
        if frame.detectionsUnion.which() != "detectionsDP":
            continue
        for det in frame.detectionsUnion.detectionsDP:
            if count >= n: break
            bits = list(det.decodedId)
            print(f"  frame={frame.frameIdx} xpos={det.xpos:5d} ypos={det.ypos:5d} "
                  f"zRot={det.zRotation:+.2f} sal={det.localizerSaliency:.3f} id_bits={bits[:4]}...")
            count += 1

def sample_polo(out_dir, label, n=5):
    parquet_files = sorted(glob.glob(os.path.join(out_dir, "*.parquet")))
    if not parquet_files:
        print(f"[{label}] no .parquet files"); return
    print(f"\n[{label}] first {n} rows from {os.path.basename(parquet_files[0])}")
    df_p = pd.read_parquet(parquet_files[0])
    print(f"  columns: {list(df_p.columns)}")
    print(df_p.head(n).to_string(index=False))

sample_hd  (OUT["hd_conda"],    "hd_conda")
sample_polo(OUT["polo_conda"],  "polo_conda")
sample_hd  (OUT["hd_docker"],   "hd_docker")
sample_polo(OUT["polo_docker"], "polo_docker")